# Build GPT from Scratch

In [2]:
# import module
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(1337);

## Data Preprocessing

In [3]:
# import tiny shakespare
with open("../input.txt", 'r', encoding="UTF-8") as f:
    text = f.read()

print(len(text))
print(text[:80])

1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.


In [4]:
# get all the characters in 'input.txt'
chars = sorted(set(text))
vocab_size = len(chars)

print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [5]:
# simple tokenizer:
# just simply treat a character as a token,
# and encode these character as number 0~65.

# character map
stoi = {ch:i for i, ch in enumerate(chars)}
itos = {i:ch for ch, i in stoi.items()}

# tokenizer
# Using lambda is for a quick call when we want to en/de-code a sequence.
encode = lambda s: [stoi[ch] for ch in s]         # endcode : take a string, return a list of integer
decode = lambda l: ''.join([itos[i] for i in l])  # decode  : take a list of integer, return a string

In [6]:
# encode whole input text

# Use torch.long so token IDs work directly as embedding indices and cross-entropy class targets.
# Embedding api requires int32 or long/int64 dtype. Cross-entropy exactly requires long/int64.
data = torch.tensor(encode(text), dtype=torch.long)

print(data.shape, data.dtype)
print(data[:80])

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8])


In [7]:
# split up data set to training and validation sets
n = int(0.9 * len(data))
tr_data = data[:n]
val_data = data[n:]

## Transformer

### input

In [8]:
# input context test
block_size = 8

x = tr_data[:block_size]
y = tr_data[1:block_size+1]

# We use different length of inputs with upper limit.
# So that whatever you input (length must under upper limit),
# the model will accept it and give you a predicition.
for i in range(block_size):
    a = (itos[j] for j in x[:i+1].tolist())
    b = itos[y[i].item()]
    print(f"when input {repr(''.join(a))}, the target will be {repr(b)}")

when input 'F', the target will be 'i'
when input 'Fi', the target will be 'r'
when input 'Fir', the target will be 's'
when input 'Firs', the target will be 't'
when input 'First', the target will be ' '
when input 'First ', the target will be 'C'
when input 'First C', the target will be 'i'
when input 'First Ci', the target will be 't'


In [9]:
# build mini-batch for batched training
batch_size = 4    # batch dimension (the number of slices in one batch, the number of parallel inputs meanwhile)
block_size = 8    # time/token dimension (the max length of a sequence of tokens)


def get_batch():
    # Don't forget minus block_size on upper limit.
    ix = torch.randint(0, len(tr_data) - block_size, (batch_size,))
    # Pytorch doesn't support by using tensor and integer to index a slice of a tensor.
    # (tensor[tensor: integer] is not suppoted.)
    # We can use comprehension to create every batch in a list,
    # then use torch.stack to stack these tensors to a big tensor.
    x = torch.stack([tr_data[i:i+block_size] for i in ix])
    y = torch.stack([tr_data[i+1:i+1+block_size] for i in ix])
    return x, y

xb, yb = get_batch()


print(f"xb shape:{xb.shape}")
print(f"xb: {xb}")
print(f"yb shape:{yb.shape}")
print(f"yb: {yb}")
for i in range(batch_size):
    for j in range(block_size):
        a = (itos[j] for j in xb[i, :j+1].tolist())
        b = itos[yb[i, j].item()]
        print(f"when input {repr(''.join(a))}, the target will be {repr(b)}")

xb shape:torch.Size([4, 8])
xb: tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
yb shape:torch.Size([4, 8])
yb: tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
when input 'L', the target will be 'e'
when input 'Le', the target will be 't'
when input 'Let', the target will be "'"
when input "Let'", the target will be 's'
when input "Let's", the target will be ' '
when input "Let's ", the target will be 'h'
when input "Let's h", the target will be 'e'
when input "Let's he", the target will be 'a'
when input 'f', the target will be 'o'
when input 'fo', the target will be 'r'
when input 'for', the target will be ' '
when input 'for ', the target will be 't'
when input 'for t', the target will be 'h'
when input 'for th', the target will be 'a'
when inpu

### bigram

In [31]:
# create bigram model by using nn.Module
class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        self.bi_emb = nn.Embedding(vocab_size, vocab_size)

    # __call__ will call forward here to perform the forward pass
    def forward(self, x, target=None):
        logits = self.bi_emb(x)  # shape: (B, T, C)
        
        if target is None:
            loss = None
        else:
            # cross_entropy requires shape(B, C, T) in 3 dimensions or shape(B, C) in 2 dimensions.
            # C (the channel/embedding dimension) must be the second dimension.
            # We need reshape our tensors.

            # flatten way
            # logits = logits.view(-1, logits.shape[2])
            # target = target.view(-1)
            # transpose way
            logits = logits.transpose(1, 2)
            
            loss = F.cross_entropy(logits, target)
        return logits, loss

    def generate(self, sq, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, loss = self(sq)
            # Focus on the last prediction, because the predictions in context has already generated.
            # We need the last prediction to predict what is the next token should be.
            logits = logits[:, -1, :]  # return a tensor with shape(B, C)
            probs = F.softmax(logits, dim=-1) # (B, C)
            ix = torch.multinomial(probs, num_samples=1) # (B, 1)
            sq = torch.cat((sq, ix), dim=1)  # (B, T+1)
        return sq

In [32]:
# run this bigram model
m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
sq = m.generate(torch.zeros((1, 1), dtype=torch.long), 15)
print(f"shape of logits: {tuple(logits.shape)}, loss: {loss.item():.4f}")
print(decode(sq[0].tolist()))

shape of logits: (32, 65, 8), loss: 4.5883

,X?Op3GONWileDj


### optimizer

An optimizer updates trainable parameters using the gradients computed during backpropagation. It does not compute gradients itself.

We can easily use optimizer to update parameters.

```python
optimizer.zero_grad(set_to_none=True)  # Clear gradients from the previous step
loss.backward()                        # Compute gradients
optimizer.step()                       # Update model parameters
```

#### SGD (stochastic gradient descent) in makemore

The makemore notebooks implement vanilla gradient descent manually:

```python
for p in parameters:
    p.grad = None

loss.backward()

for p in parameters:
    p.data += -learning_rate * p.grad
```

This applies the standard update rule:

$$
\theta \leftarrow \theta - \eta \nabla_\theta L
$$

When the gradient is estimated from a randomly sampled mini-batch, this becomes mini-batch stochastic gradient descent (SGD). The neural bigram model uses full-batch gradient descent, while later makemore models use mini-batch SGD.

In [33]:
# SGD
# optimizer = torch.optim.SGD(m.parameters(), lr=lr)

#### AdamW

AdamW maintains exponential moving averages of both gradients and squared gradients, allowing it to adapt the update scale for each parameter. It also applies weight decay independently from the gradient-based update.

```python
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate,
)
```

Compared with the manual SGD implementation, AdamW manages its internal optimization state and parameter updates automatically. It is commonly used for training Transformer-based models.

In [34]:
# create a pytorch optimizer AdamW
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

### train

In [35]:
batch_size = 32

for steps in range(20000):
    # mini-batch
    xb, yb = get_batch()
    
    # forward pass
    logits, loss = m(xb, yb)
    
    # backward pass
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    
    # update
    optimizer.step()

print(loss.item())

2.4368038177490234


In [36]:
# generate something after training
sq = m.generate(torch.zeros((1, 1), dtype=torch.long), 300)
print(decode(sq[0].tolist()))


MENISourees m fr m?
DWhafin s f achang h.'sllm bu s he thanoufulltoth lileshane tedes hallorist, ourlawr peavere byoupespe, w d mee yoanakn, so hip afogller-CLARMERDItee the yne fung USo tie, al CEd cilar.
K:
Fard tealy orth oy l t lad: ar thathad hy.

Tougo om.
The, beme I:
MOn menge thar linghand 
